# 4-01 장마감 브리프 — 지수·종목동향·상위 종목·매크로를 한 장으로

**질문 예시:** "어제 장 마감 시황 정리해줘", "11월 24일 코스피·코스닥 마감 브리프 만들어줘"

CHECK 원본(`4_시황재현/01_장마감브리프`)은 KRX Brief를 **지수·지수선물·투자자 수급 + 금리·환율·유가**로 재현했습니다.
DeepSearch에서는 아래 구성으로 바꿉니다.

| 브리프 항목 | DeepSearch | 비고 |
|---|---|---|
| 지수(코스피·코스닥 계열 11종) | `GetMarketIndexes` | 그대로 |
| 상승·하락·보합·상한가·하한가 종목 수 | `GetStockPrices` 전 종목 하루치 | 직접 집계 (상·하한가는 ±29.5% 근사) |
| 시가총액·거래대금 상위 종목 | 같은 전 종목 스냅샷 | CHECK에 없던 추가 항목 |
| 금리·환율·유가·해외지수 | FRED (`QueryFredSeriesData`) | 다우·나스닥·브렌트유도 됨 (CHECK에선 범위 밖) |
| 오늘 시장 뉴스 | `DocumentSearch`, `DocumentAggregation` | CHECK에 없던 추가 항목 |
| **지수선물, 투자자별 수급, 국고채** | **없음** | DeepSearch에 데이터가 없음 (`research/data_gaps`) |

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/4_market_brief/01_closing_brief.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"   # 브리프 날짜 (그날이 휴장이면 직전 거래일)

## 1. 지수
11종 지수의 종가·등락·거래대금입니다. 지수 `change_rate`는 **퍼센트**, `value`(거래대금)는 **백만원** 단위입니다
(코스피 거래대금 = 개별 종목 거래대금 합과 같은 규모인지 아래 3절에서 검산합니다).

In [4]:
d = pd.Timestamp(ASOF)
ix = index_prices((d - pd.Timedelta(days=10)).date(), d.date())
DAY = ix["date"].max()
print("브리프 기준 거래일:", DAY.date())
ORDER = ["KRX:KOSPI", "KRX:KOSPI200", "KRX:KOSPI100", "KRX:KOSPI50", "KRX:KOSPILC", "KRX:KOSPIMC", "KRX:KOSPISC",
         "KRX:KOSDAQ", "KRX:KOSDAQLC", "KRX:KOSDAQMC", "KRX:KOSDAQSC"]
t = ix[ix["date"] == DAY].set_index("symbol").loc[[s for s in ORDER if s in set(ix["symbol"])]]
idx_table = pd.DataFrame({
    "지수": t["entity_name"],
    "종가": t["close"].round(2),
    "전일 대비": t["change"].round(2),
    "등락률(%)": t["change_rate"].round(2),
    "거래대금": (t["value"] * 1e6).map(krw),
}).reset_index(drop=True)
idx_table

브리프 기준 거래일: 2026-11-24


,지수,종가,전일 대비,등락률(%),거래대금
0,코스피,5006.03,297.49,6.32,15.8조
1,코스피 200,778.68,47.85,6.55,14.3조
2,코스피 100,6057.43,376.86,6.63,13.3조
3,코스피 50,5855.31,370.76,6.76,11.9조
4,코스피 대형주,5401.27,335.34,6.62,13.7조
5,코스피 중형주,3423.71,121.60,3.68,1.2조
6,코스피 소형주,2069.46,50.28,2.49,"2,876억"
7,코스닥,665.88,30.43,4.79,5.9조
8,코스닥 대형주,1583.30,87.00,5.81,3.3조
9,코스닥 중형주,733.56,30.74,4.37,1.3조


## 2. 종목동향 — 상승·하락·보합·상한가·하한가
전 종목 하루치 시세를 한 번에 받아 시장별로 셉니다. ETF·ETN 등(`company_type_l1`=8)은 빼고, 우선주는 넣었습니다.
상한가·하한가는 가격제한폭(±30%)에 닿은 종목을 **등락률 ±29.5~30.5%**로 근사합니다. ±30%를 넘는 종목은 신규상장 첫날이거나
권리락·감자 등으로 기준가가 바뀐 날일 가능성이 커서 따로 셉니다(예: 2026-11-24 엠비디 −39.6%는 상장 첫날).

In [5]:
snap = all_stocks(DAY, common_only=False)
snap = snap[snap["company_type_l1"] != "8"]
snap = snap[snap["date"] == DAY]
traded = snap[snap["volume"] > 0]

def breadth(df):
    r = df["change_rate"]
    return pd.Series({"종목 수": len(df), "상승": (r > 0).sum(), "하락": (r < 0).sum(), "보합": (r == 0).sum(),
                      "상한가(근사)": ((r >= 0.295) & (r <= 0.305)).sum(), "하한가(근사)": ((r <= -0.295) & (r >= -0.305)).sum(),
                      "±30% 초과": (r.abs() > 0.305).sum()})

bt = pd.DataFrame({m: breadth(traded[traded["market"] == m]) for m in ["KOSPI", "KOSDAQ"]}).T
print("거래 없는 종목(정지 등) 제외:", len(snap) - len(traded), "종목")
bt

거래 없는 종목(정지 등) 제외: 145 종목


,종목 수,상승,하락,보합,상한가(근사),하한가(근사),±30% 초과
KOSPI,910,797,94,19,2,0,0
KOSDAQ,1750,1389,322,39,10,0,1


## 3. 시가총액·거래대금
시장별 합계와, 지수 거래대금(백만원 단위)과의 검산입니다. 규모는 맞지만 **지수 거래대금이 종목 합보다 7~9% 큽니다**
(2026-11-24 코스피 15.8조 vs 14.7조). 시간외 거래 포함 여부 같은 정의 차이로 보이나 확인하지 못했습니다.
시장 거래대금을 말할 때는 어느 쪽 숫자인지 밝히세요.

In [6]:
mk = snap.groupby("market").agg(시가총액=("market_cap", "sum"), 거래대금=("value", "sum")).loc[["KOSPI", "KOSDAQ"]]
idx_val = t.set_index("entity_name")["value"] * 1e6
mk["지수 거래대금"] = [idx_val.get("코스피"), idx_val.get("코스닥")]
mk.apply(lambda c: c.map(krw))

,시가총액,거래대금,지수 거래대금
market,,,
KOSPI,4104.8조,14.7조,15.8조
KOSDAQ,381.4조,5.4조,5.9조


## 4. 시가총액 상위 10 · 거래대금 상위 10

In [7]:
common = snap[snap["symbol"].str.endswith("0") & (snap["company_type_l1"] == "1")]

def show(df):
    return pd.DataFrame({"종목": df["name"], "시장": df["market"], "종가": df["close"].map("{:,.0f}".format),
                         "등락률(%)": (df["change_rate"] * 100).round(2), "시가총액": df["market_cap"].map(krw),
                         "거래대금": df["value"].map(krw)}).reset_index(drop=True)

print(PRICE_NOTICE)
print("\n[시가총액 상위 10]")
display(show(common.nlargest(10, "market_cap")))
print("[거래대금 상위 10]")
display(show(common.nlargest(10, "value")))

※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). 거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).

[시가총액 상위 10]


,종목,시장,종가,등락률(%),시가총액,거래대금
0,삼성전자,KOSPI,"189,400",6.70,1107.3조,3.0조
1,SK하이닉스,KOSPI,"1,158,000",6.73,818.5조,3.9조
2,SK스퀘어,KOSPI,"715,000",5.15,94.3조,"4,165억"
3,LG에너지솔루션,KOSPI,"314,500",7.89,73.6조,"1,208억"
4,삼성전기,KOSPI,"858,000",10.28,64.1조,"6,675억"
5,삼성바이오로직스,KOSPI,"1,290,000",0.16,59.7조,829억
6,현대차,KOSPI,"275,000",11.11,56.0조,"1,842억"
7,KB금융,KOSPI,"151,400",5.36,53.7조,"1,049억"
8,삼성생명,KOSPI,"247,000",9.78,49.4조,591억
9,셀트리온,KOSPI,"209,000",3.47,48.5조,784억


[거래대금 상위 10]


,종목,시장,종가,등락률(%),시가총액,거래대금
0,SK하이닉스,KOSPI,"1,158,000",6.73,818.5조,3.9조
1,삼성전자,KOSPI,"189,400",6.70,1107.3조,3.0조
2,삼성전기,KOSPI,"858,000",10.28,64.1조,"6,675억"
3,SK스퀘어,KOSPI,"715,000",5.15,94.3조,"4,165억"
4,원익홀딩스,KOSDAQ,"31,350",-2.18,2.4조,"3,446억"
5,삼성SDI,KOSPI,"450,000",13.64,36.3조,"3,193억"
6,로보티즈,KOSDAQ,"294,000",19.51,4.3조,"2,090억"
7,주성엔지니어링,KOSDAQ,"149,800",5.72,7.0조,"2,026억"
8,현대차,KOSPI,"275,000",11.11,56.0조,"1,842억"
9,LG전자,KOSPI,"143,200",15.86,23.3조,"1,831억"


## 5. 기타 — 금리·환율·유가·해외지수 (FRED)
FRED 미국 시리즈는 공표가 1~3 영업일 늦습니다. 그래서 각 시리즈의 **기준일을 함께** 보여 줍니다.
등락은 바로 앞 관측치 대비입니다.

In [8]:
MACRO = {"DGS2": ("미국 2년 국채", "%"), "DGS10": ("미국 10년 국채", "%"), "DEXKOUS": ("원/달러", "원"),
         "DTWEXBGS": ("달러지수(광의)", ""), "DCOILWTICO": ("WTI", "$"), "DCOILBRENTEU": ("브렌트유", "$"),
         "DJIA": ("다우존스", ""), "NASDAQCOM": ("나스닥 종합", ""), "SP500": ("S&P 500", "")}
mac = fred(list(MACRO), (d - pd.Timedelta(days=20)).date(), d.date())
rows = []
for sid, (name, unit) in MACRO.items():
    s = mac[sid].dropna() if sid in mac else pd.Series(dtype=float)
    if len(s) < 2:
        rows.append({"항목": name, "시리즈": sid, "기준일": None, "값": None, "직전 대비": None})
        continue
    chg = s.iloc[-1] - s.iloc[-2]
    rows.append({"항목": name, "시리즈": sid, "기준일": s.index[-1].date(), "값": round(s.iloc[-1], 2),
                 "직전 대비": f"{chg:+.2f}" + ("%p" if unit == "%" else f" ({chg / s.iloc[-2] * 100:+.2f}%)")})
pd.DataFrame(rows)

,항목,시리즈,기준일,값,직전 대비
0,미국 2년 국채,DGS2,2026-11-20,4.63,-0.13%p
1,미국 10년 국채,DGS10,2026-11-20,4.92,-0.12%p
2,원/달러,DEXKOUS,2026-11-20,1342.41,+9.97 (+0.75%)
3,달러지수(광의),DTWEXBGS,2026-11-20,123.60,+0.07 (+0.06%)
4,WTI,DCOILWTICO,2026-11-17,89.53,+0.29 (+0.32%)
5,브렌트유,DCOILBRENTEU,2026-11-17,92.94,+0.04 (+0.04%)
6,다우존스,DJIA,2026-11-23,49595.31,+1447.98 (+3.01%)
7,나스닥 종합,NASDAQCOM,2026-11-19,23879.20,-550.68 (-2.25%)
8,S&P 500,SP500,2026-11-23,7237.17,+326.78 (+4.73%)


## 6. 오늘 시장 뉴스
기준일 경제 뉴스 중 증시 관련 제목과, 그 기사들에 종목 태그가 가장 많이 붙은 종목입니다.
종목 태그는 DeepSearch가 기사에서 자동 인식한 것이라 **오인식이 섞일 수 있습니다**
(예: 2026-11-24에는 '엔비디아' 기사들이 코스닥 종목 '엠비디'로 태그되어 1위로 나왔습니다). 상위 종목은 제목으로 한 번 확인하세요.

In [9]:
total, news = doc_search("news", ["economy"], "title:(코스피 or 코스닥 or 증시)", DAY.date(), DAY.date(), count=100)
print(f"증시 제목 기사 {total:,}건 (최근 10건)")
display(news.sort_values("created_at", ascending=False).head(10)[["created_at", "publisher", "title"]])
agg = ds_df(f'DocumentAggregation(["news"],["economy"],"title:(코스피 or 코스닥 or 증시)","securities.name:10",'
            f'date_from={DAY.date()},date_to={DAY.date()})')
agg = agg.rename(columns={"key": "종목", "count": "기사 수"})[["종목", "기사 수"]]
print("증시 기사에 많이 나온 종목")
agg

증시 제목 기사 539건 (최근 10건)


,created_at,publisher,title
0,2026-11-24 22:03:11,경기일보,‘명픽’ 이광수의 ‘코스피 5000 붕괴 눈물’...그 영상 공유한 조국
1,2026-11-24 22:00:00,이데일리,"""5000피 탈환 코스피, 바닥 다졌나?""…전문가 판단은"
2,2026-11-24 21:45:00,경향신문,사흘 만에 코스피 5000선 탈환…불안 요인 해소는 아직
3,2026-11-24 21:41:00,헤럴드경제,저PBR 잡겠다 공언했는데…코스피 PBR 결국 작년으로 회귀 [투자360]
4,2026-11-24 21:20:00,TV조선,"""AI 붐 위해 개입""…엔비디아 31% 폭등하며 세계 증시 견인"
5,2026-11-24 21:03:00,KBS,젠슨 황 복귀에 코스피 급등…5천선 회복
6,2026-11-24 20:59:00,한국경제TV,증시 반색한 'AI 황제' 깜짝 귀환…남은 변수에 '신중론'도
7,2026-11-24 20:34:00,서울신문,‘증안펀드 카드’도 못 살린 코스피… ‘황타랠리’에 5000선 회복
8,2026-11-24 20:26:10,경기일보,코스닥 흔들리자 지역상장사도 ‘온도차’…공시로 본 시장 변화 [경기·인천 코스닥을 묻다②]
9,2026-11-24 20:21:00,경기일보,코스닥 37% 경기·인천…대한민국 성장의 심장 [경기·인천 코스닥을 묻다①]


증시 기사에 많이 나온 종목


,종목,기사 수
0,엠비디,12
1,SK하이닉스,8
2,삼성전자,7
3,키움증권,5
4,NAVER,2
5,영광,2
6,LS증권,1
7,NH투자증권,1
8,대신증권,1
9,머니무브,1


## 7. 한 줄 요약 (자동 문장)

In [10]:
k = t.set_index("entity_name")
kp, kq = bt.loc["KOSPI"], bt.loc["KOSDAQ"]
top = common.nlargest(1, "market_cap").iloc[0]
print(f"{DAY.date()} 코스피는 {k.loc['코스피','close']:,.2f}({k.loc['코스피','change_rate']:+.2f}%), "
      f"코스닥은 {k.loc['코스닥','close']:,.2f}({k.loc['코스닥','change_rate']:+.2f}%)로 마감했습니다. "
      f"코스피는 {kp['상승']:,}종목 상승·{kp['하락']:,}종목 하락, 코스닥은 {kq['상승']:,}종목 상승·{kq['하락']:,}종목 하락입니다. "
      f"시가총액 1위 종목({top['name']}) 등락률은 {top['change_rate'] * 100:+.2f}%입니다.")
print("※ 투자자별 수급(외국인·기관·개인)과 지수선물은 DeepSearch에 데이터가 없어 이 브리프에 넣지 못했습니다.")

2026-11-24 코스피는 5,006.03(+6.32%), 코스닥은 665.88(+4.79%)로 마감했습니다. 코스피는 797종목 상승·94종목 하락, 코스닥은 1,389종목 상승·322종목 하락입니다. 시가총액 1위 종목(삼성전자) 등락률은 +6.70%입니다.
※ 투자자별 수급(외국인·기관·개인)과 지수선물은 DeepSearch에 데이터가 없어 이 브리프에 넣지 못했습니다.
